In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"font.size": 10, "svg.fonttype": "path"})

# named colors from the course palette
BLUE, VERMILLION, GREEN, PINK, GRAY = "#0072B2", "#D55E00", "#009E73", "#cb77a6", "#7f7f7f"

This section draws heavily on chapter 2, Configuration Space, of Kevin Lynch and Frank Park, *Modern Robotics: Mechanics, Planning, and Control* (Cambridge University Press, 2017). You can [download the chapter as a PDF](https://artifacts.aegeanai.com/pdf/lynch-mr/config-space.pdf).

A robot consists of bodies called links, which are connected by joints. Actuators, such as electric motors, apply forces or torques that move the links. An end-effector, such as a gripper or hand, is usually attached to one link for grasping and manipulating objects.

A robot’s _configuration_ specifies the positions of all points in its mechanisms.

Consider a coin lying heads up on a table. Such an object is a _planar_ rigid body. A _spatial_ rigid body can move anywhere in three-dimensional space. Three real numbers, also called coordinates, describe the coin's configuration. Two coordinates $(x,y)$ locate a chosen point on the coin, and one coordinate ($θ$) specifies its orientation. If the coin may also lie tails up, the visible face is part of its configuration. This value belongs to the discrete set $\{\text{heads}, \text{tails}\}$ rather than a continuous range of real numbers.

![Configuration of a coin on a flat surface.](images/coin.png)

You can place the coin anywhere on the table, which gives two degrees of freedom for the $(x,y)$ coordinates. The point alone does not fix the coin's configuration. If the chosen point is Abraham Lincoln's eye, for example, the coin can still rotate through 360 degrees. The angle $\theta$ describes this additional degree of freedom.

The number of degrees of freedom (dof) of a robot is the _smallest_ number of real-valued coordinates needed to represent its configuration. The discrete choice of face adds no degree of freedom. The coin has $f=3$ degrees of freedom. Its current configuration is one point in the space of all its possible configurations, called the _configuration space_ (C-space).

Lift the coin off the table and let it move anywhere in space. Choose three points $A$, $B$ and $C$ on its rim. Point $A$ can be anywhere, so it takes three coordinates. Once $A$ is held, $B$ keeps its distance to $A$ and lies on a sphere centered at $A$, which adds two coordinates. Once $A$ and $B$ are held, $C$ keeps its distance to both. It lies on a sphere centered at $A$ and on a sphere centered at $B$, and the two spheres meet in a circle. One angle places $C$ on that circle. The coin in space has $3 + 2 + 1 = 6$ degrees of freedom.

<video
  src="https://artifacts.aegeanai.com/manim/configuration-space/CoinThreeSpheres.mp4?v=3"
  controls
  loop
  muted
  playsInline
  width="100%"
  style={{ borderRadius: 8 }}
/>

*The coin first moves freely, and $A$ traces its path. With $A$ held, $B$ stays on a sphere about $A$. With $A$ and $B$ held, the camera turns so that the line $AB$ runs across the screen. The sphere about $A$ (red) and the sphere about $B$ (blue) meet in the yellow circle, and the coin turns about $AB$ while $C$ runs around that circle. The coin then lies flat on a table. $A$ moves in the plane, which takes two coordinates. With $A$ held, $B$ stays on a circle about $A$, which adds one. With $A$ and $B$ held, the circle about $A$ (red) and the circle about $B$ (blue) meet in only two points. Flipping the coin from heads to tails moves $C$ from one point to the other, so $C$ adds a choice but no coordinate. The coin on the table has $2 + 1 = 3$ degrees of freedom.*

A large drone, treated as a single rigid body for navigation, has six degrees of freedom: three Cartesian coordinates and three angles.

![Heading, pitch and roll rotations of an aircraft relative to a North-East-Down reference frame](images/yaw-pitch-roll.png)

Pitch, yaw (heading), and roll describe rotations about the principal axes of a drone located at point $\mathbf p = (x,y,z)^T$. Figure by the [Motion Imagery Standards Board](https://commons.wikimedia.org/wiki/File:MISB_ST_0601.8_-_Yaw,_Pitch_%26_Roll.png), public domain.

The coordinates used for the drone are one possible _configuration representation_. Other representations have other strengths. Six degrees of freedom are enough for navigation. For kinematic or dynamic analysis you must also count the moving parts, such as the flaps, because they affect the motion. A robot generally consists of several rigid bodies whose motions are constrained by its mechanical structure. Its degrees of freedom are given by

$$
\mathrm{dof} = \text{(freedoms of the bodies)} - \text{(independent constraints)}
$$

Joints constrain the motion of rigid bodies and reduce their total degrees of freedom.

![Revolute, prismatic, helical, cylindrical, universal and spherical joints.](images/joints.png)

The revolute joint (R), also called a hinge joint, allows rotation about the joint axis. The hinges of a door are revolute joints.

The prismatic joint (P), also called a sliding or linear joint, allows translation along the joint axis. A sliding door moves on one.

The helical joint (H), also called a screw joint, allows simultaneous rotation and translation about a screw axis. Revolute, prismatic and helical joints each have one degree of freedom.

Some joints have multiple degrees of freedom. The cylindrical joint (C) has two degrees of freedom. It allows independent translation and rotation about one fixed joint axis.

The universal joint (U) is another two-degree-of-freedom joint. It consists of two revolute joints with orthogonal axes.

The spherical joint (S), also called a ball-and-socket joint, has three degrees of freedom. Your shoulder works like one.

The following table gives the dof of each joint when it connects planar or spatial rigid bodies.

| Joint type         | DoF  | Constraints between two planar rigid bodies | Constraints between two spatial rigid bodies |
|--------------------|---------|-----------------------------------------------|--------------------------------------------------|
| Revolute (R)       | 1       | 2                                             | 5                                                |
| Prismatic (P)      | 1       | 2                                             | 5                                                |
| Helical (H)        | 1       | N/A                                           | 5                                                |
| Cylindrical (C)    | 2       | N/A                                           | 4                                                |
| Universal (U)      | 2       | N/A                                           | 4                                                |
| Spherical (S)      | 3       | N/A                                           | 3                                                |


_Grübler's rule_ gives the degrees of freedom of a robot:

$$
\mathrm{dof} = m \left( N - 1 - J \right) + \sum_{i=1}^{J} f_i
$$

where $m = 3$ for planar mechanisms and $m = 6$ for spatial mechanisms, $N$ is the number of links (including the ground link), $J$ is the number of joints, and $f_i$ is the number of degrees of freedom of joint $i$. The formula assumes that the constraints imposed by the joints are independent. When they are dependent, as can occur in closed chains, the formula gives only a lower bound on the degrees of freedom.

## Example: the human arm

How many degrees of freedom does your arm have? Count from your torso to your palm. Keep the center of your shoulder fixed and exclude your fingers.

<img src="images/human-arm.png" alt="Human arm with its shoulder, elbow, and wrist joints marked." width="278" />

*The arm as a chain of four links: torso, upper arm, forearm and palm.*

Each joint of the arm corresponds to a type in the table above:

| Joint of the arm | Modeled as | dof $f$ | Constraints $c$ |
|--------------------|---------------|---------|-----------------|
| Shoulder | Spherical (S) | 3 | 3 |
| Elbow and forearm | Universal (U): bending, and turning the palm up or down | 2 | 4 |
| Wrist | Universal (U): bending up and down, tilting toward the thumb or the little finger | 2 | 4 |

The four links give $N = 4$, and the three joints give $J = 3$. The joint freedoms sum to $3 + 2 + 2 = 7$. Grübler's rule with $m = 6$ gives

$$
\mathrm{dof} = 6 \left( 4 - 1 - 3 \right) + 7 = 7 .
$$

## Configuration space topology

Consider a point moving on the surface of a sphere. Its C-space is two-dimensional because two coordinates, latitude and longitude, describe its configuration. A point moving on a plane also has a two-dimensional C-space, with coordinates $(x, y)$. The two spaces have different shapes. The plane extends forever, while the sphere wraps around.

An oval-shaped American football wraps around like a sphere. You can obtain its shape by stretching a sphere in one direction.

_Topology_ describes the sense in which the surfaces of a small sphere, a large sphere and a football have the same kind of shape, while a plane has a different shape. The figure shows several example topologies.

![Examples of C-space topologies.](images/configuration-topologies.png)

In the figure, $S^2$ is the two-dimensional surface of a sphere in three-dimensional space.

The C-space of a rigid body in the plane is $\mathbb{R}^2 \times S^1$. The coordinates $(x, y)$ form $\mathbb{R}^2$, and the angle $\theta$ forms $S^1$.

The C-space of a 2R robot arm is $S^1 \times S^1 = T^2$, where $T^n$ is the $n$-dimensional surface of a torus in an $(n+1)$-dimensional space. The product $S^1 \times S^1 \times \cdots \times S^1$ (with $n$ copies of $S^1$) equals $T^n$, not $S^n$. For example, a sphere $S^2$ is not topologically equivalent to a torus $T^2$.

The topology of a space is a property of the space itself. It does not depend on the coordinates used to represent its points. You can represent a point on a circle by the angle $\theta$ from a chosen zero angle. You can instead place the origin of a reference frame at the circle's center and use two coordinates $(x, y)$ subject to the constraint $x^2 + y^2 = 1$. The choice of coordinates does not change the space.

## C-space representation

A choice of $n$ coordinates for an $n$-dimensional space is an _explicit parametrization_ of the space. For a point on a sphere, the familiar choice is the pair used for navigation on Earth:

* Latitude (ϕ): angle from the equator (−90° at South Pole to +90° at North Pole)
* Longitude (λ): angle around the equator (from −180° to +180°)

This coordinate system works well except at the poles. Suppose you are close to the North Pole (ϕ ≈ 90°) and take a small step eastward. Your latitude ϕ changes little, but your longitude λ may change greatly. It may even jump from +180° to −180° after a small move.

A _singularity_ occurs when a small physical motion causes a large coordinate change. The sphere remains smooth and uniform, so the singularity belongs to the chosen representation. Such singularities cause problems when computing velocities. Coordinate rates such as $\dot\phi$ and $\dot\lambda$ can grow very large or diverge near the poles even when the actual speed in three-dimensional space is constant.

<video
  src="https://artifacts.aegeanai.com/manim/configuration-space/SphereLatLongSingularity.mp4?v=3"
  controls
  loop
  muted
  playsInline
  width="100%"
  style={{ borderRadius: 8 }}
/>

*The point $P$ illustrates the coordinates. Longitude $\lambda$ runs around the equator from the prime meridian, while latitude $\phi$ runs from the equator toward a pole. A point then moves at constant speed along a great circle that passes 6° from the North Pole. During this pass, it crosses half the longitude range on the latitude-longitude chart, and $|\dot\lambda|$ rises sharply. The view from above explains this behavior. A small circle around the pole crosses every longitude, while the chart's entire top edge $\phi = 90^\circ$ represents the pole itself.*

The video below shows the same kind of singularity on a robot arm.

<iframe width="560" height="315" src="https://www.youtube.com/embed/93Grb3Pxido" frameBorder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowFullScreen></iframe>

{/*
The robot in the video meets the same problem the sphere has at its poles. Place the wrist center of the arm at cylindrical coordinates $(r, \theta_1, z)$ around the axis of the first joint. The first joint angle is the polar angle of the wrist center,

$$
\theta_1 = \operatorname{atan2}(y, x),
$$

and on the axis itself, where $r = 0$, it is not defined at all. This is longitude at the north pole.

Now let the tool draw a straight line at constant speed $v$, passing the axis at a distance $d$. Put the line along $x$, so $y = d$ throughout. Differentiating $\theta_1$ gives the joint speed

$$
\dot\theta_1 = -\frac{v \, d}{x^2 + d^2}.
$$

The sign only gives the direction of the turn. The speed $|\dot\theta_1|$ peaks at $v / |d|$ as the tool passes closest to the axis. Halve the distance and the joint must turn twice as fast. At $d = 0$ the joint would have to swing $180^\circ$ in no time at all.

### Why this is dangerous for the motors

The tool speed is what you program, but the motors pay for it in joint speed and joint torque. Near a singularity the arm needs very large joint motion to produce ordinary tool motion, and that goes wrong in several ways.

- **Overspeed.** Every motor and gearbox has a maximum speed $\omega_{\max}$. Past it the controller either faults and stops the robot, or slows the whole path down, so the tool no longer moves at the speed you programmed.
- **Overtorque.** The joint must speed up and slow down within a short stretch of the path. Torque grows with angular acceleration, and here the acceleration grows like $v^2 / d^2$. When the demand exceeds what the motor can deliver, the arm falls behind its path, draws excess current, and heats its windings.
- **Sudden flips.** The joint idles for most of the pass and then whips through close to $180^\circ$ at once. That is a shock load on the gearbox, a strain on cables and tooling, and a hazard to anyone standing near the arm.

The danger is a region, not a point. For a pass with $d > 0$, setting $|\dot\theta_1| > \omega_{\max}$ and completing the square gives

$$
x^2 + \left( d - \frac{v}{2\omega_{\max}} \right)^2 < \left( \frac{v}{2\omega_{\max}} \right)^2 ,
$$

a disk of diameter $v / \omega_{\max}$ that touches the axis. Lines on the other side of the axis give its mirror image. Together the two disks form the figure-eight shape the video maps. Its size is proportional to the tool speed, so the faster you program the robot, the larger the region you have to keep the wrist out of. */}

You can avoid such singularities with an _implicit representation_. This representation treats an $n$-dimensional space as _embedded_ in a Euclidean space of more than $n$ dimensions. For example, a two-dimensional unit sphere is a surface embedded in three-dimensional Euclidean space. An implicit representation uses the coordinates of the higher-dimensional space, such as $(x, y, z)$, with constraints that reduce the number of degrees of freedom. For the unit sphere, the constraint equation is:

$$
x^2 + y^2 + z^2 = 1
$$

An implicit representation uses more numbers than there are degrees of freedom, but it has no singularities. Rotation matrices are the main example in this chapter. A rotation matrix describes an orientation in space with nine numbers subject to six constraints. It also acts on vectors as a linear operator, so you can work with it using matrix algebra.

### Velocity constraints: the TurtleBot

A configuration space describes where a robot can be. A wheeled robot cannot move in every direction from each configuration. Lynch and Park describe these limits as velocity constraints in section 2.4 [@Kevin-M-Lynch2017-mp]. A differential-drive robot such as the TurtleBot provides an example.

1. Where can the robot be? Its configuration is $q = (x, y, \theta)$, which gives its position in the plane and its heading. Its configuration space is $\mathbb{R}^2 \times S^1$.
2. What motions are possible in the plane? At a configuration $q$, the velocity $\dot q = (\dot x, \dot y, \dot\theta)$ describes an instantaneous motion. A rigid body that slides freely on a table can have any combination of these components.
3. Which motions can the TurtleBot make? It drives forward at speed $v$ and turns at rate $\omega$. Forward motion follows its heading, while turning changes that heading:
   $$
   \dot q = v \begin{bmatrix} \cos\theta \\ \sin\theta \\ 0 \end{bmatrix} + \omega \begin{bmatrix} 0 \\ 0 \\ 1 \end{bmatrix} .
   $$
   The first column represents forward motion, and the second represents rotation. The velocity has three components, but the robot has only these two independent choices.
4. Which motion is forbidden? The wheels cannot slide sideways, so the velocity perpendicular to the heading must be zero. Eliminate $v$ from $\dot x = v\cos\theta$ and $\dot y = v\sin\theta$ to obtain
   $$
   -\sin\theta\,\dot x + \cos\theta\,\dot y = 0 .
   $$
   In matrix form this is $A(q)\,\dot q = 0$, with
   $$
   A(q) = \begin{bmatrix} -\sin\theta & \cos\theta & 0 \end{bmatrix} .
   $$
   A constraint written as $A(q)\dot q = 0$ is a *Pfaffian constraint*.
5. Does this reduce the configuration space? No. Although the robot cannot move sideways at an instant, it can reach a point beside its starting position by turning, driving, and turning back. This velocity constraint does not limit which configurations are reachable, so you cannot integrate it into a constraint on $q$ alone. Lynch and Park call such a constraint *nonholonomic* (book pages 31 and 32).

The TurtleBot's C-space $\mathbb{R}^2 \times S^1$ has three dimensions, but it is not $\mathbb{R}^3$. Picture it as a stack of copies of the floor, one for each heading, with $\theta$ running upward from $0$ to $2\pi$. A point in the stack is one complete configuration: the floor coordinates give the position $(x, y)$, and the height gives the heading. The heading $\theta = 2\pi$ is the same as $\theta = 0$, so the top of the stack is glued to its bottom. This is the representation $\mathbb{R}^2 \times [0, 2\pi)$, built the same way as the rotating sliding knob's $\mathbb{R}^1 \times [0, 2\pi)$ in Table 2.2 of Lynch and Park (book page 26). The gluing changes the shape of the space. A robot that turns once in place traces a closed loop through the stack, and you cannot shrink that loop to a point without leaving the space, while every loop in $\mathbb{R}^3$ shrinks to a point. The space can still be drawn in $\mathbb{R}^3$ without a seam if one line is left out. Send $(x, y, \theta)$ to the point at distance $e^x$ from a vertical axis, at angle $\theta$ around it and at height $y$. This turns $\mathbb{R}^2 \times S^1$ into $\mathbb{R}^3$ with the axis removed, and the loop of a turn in place goes around the missing axis.

<Note>

Think of each floor in the stack as a page of a book. Each page is a half-plane whose edge lies along the spine. Bending the stack opens the book all the way around, so the page for heading $\theta$ stands at angle $\theta$ around the spine. The last page, $\theta = 2\pi$, lands on the first, $\theta = 0$, and the seam disappears.

On each page, the height is $y$ and the distance from the spine is $e^x$. As $x$ decreases, the point moves toward the spine but never reaches it. No page contains the spine, so the spine is not part of the space. It is the missing axis.

A turn in place walks once around the spine and crosses every page. That loop cannot shrink to a point, because the spine is in the way.

The pages at one distance from the spine, that is one value of $x$, form the surface of a cylinder. The whole space is these cylinder surfaces nested around the spine. It is a solid cylinder with its center line removed. In a full solid cylinder, the loop could shrink through the center.

</Note>

The code below parks the robot. It starts at $q_0 = (0, 0, 0)$, facing along the $x$ axis. It drives an arc to the left, then an arc to the right, then straight back. Each step uses the motion equation from item 3.

In [ ]:
def drive(q, v, omega, duration, n=200):
    """Integrate q' = v (cos theta, sin theta, 0) + omega (0, 0, 1) from q."""
    x, y, theta = q
    dt = duration / n
    path = []
    for _ in range(n):
        x += v * np.cos(theta) * dt
        y += v * np.sin(theta) * dt
        theta += omega * dt
        path.append((x, y, theta))
    return np.array(path)


def embed(x, y, theta):
    """Send a configuration (x, y, theta) to a point of R^3 off the vertical axis."""
    r = np.exp(x)
    return r * np.cos(theta), r * np.sin(theta), y


# parallel parking: arc left by alpha, arc right by alpha, then reverse straight
rho, alpha = 0.5, np.pi / 3
q0 = np.zeros(3)
left = drive(q0, rho, 1.0, alpha)
right = drive(left[-1], rho, -1.0, alpha)
back = drive(right[-1], -1.0, 0.0, 2 * rho * np.sin(alpha))
park = np.vstack([q0, left, right, back])
qf = park[-1]
print("start (x, y, theta):", q0)
print("end   (x, y, theta):", qf.round(3) + 0.0)  # + 0.0 turns -0.0 into 0.0

The robot ends at $(0, 0.5, 0)$. It is half a meter to the left of where it started and faces the same way. It never moved sideways to get there. The figure below shows this path and a turn in place, drawn with the map.

In [ ]:
fig = plt.figure(figsize=(6.4, 5.6))
ax = fig.add_subplot(projection="3d")
ax.view_init(elev=20, azim=-30)

# the missing axis
ax.plot([0, 0], [0, 0], [-0.7, 1.2], color=GRAY, ls="--", lw=1.2)
ax.text(0, 0, 1.28, "missing axis", color=GRAY, ha="center", fontsize=9)

# all positions at heading theta = 0 form a half-plane bounded by the axis
xs, ys = np.meshgrid(np.linspace(-1.2, 1.0, 2), np.linspace(-0.7, 1.2, 2))
ax.plot_surface(*embed(xs, ys, 0.0), color=BLUE, alpha=0.12, linewidth=0)
ax.text2D(0.62, 0.16, r"all positions at heading $\theta = 0$", transform=ax.transAxes, color=BLUE, ha="center")

# turn in place at (x, y) = (0, 0): a loop around the missing axis
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(*embed(0.0, 0.0, theta), color=VERMILLION, lw=2)
ax.text2D(0.17, 0.36, "turn in place", transform=ax.transAxes, color=VERMILLION, ha="center")

# parallel parking ends straight above q0
ax.plot(*embed(park[:, 0], park[:, 1], park[:, 2]), color=GREEN, lw=2)
ax.scatter(*embed(*q0), color="black", s=22, depthshade=False)
ax.scatter(*embed(*qf), color=GREEN, s=22, depthshade=False)
ax.text(*embed(0.05, -0.14, -0.08), r"$q_0$", fontsize=11)
ax.text2D(0.8, 0.62, "parallel parking", transform=ax.transAxes, color=GREEN, ha="center")

# directions at q0: forward is outward, turning is around the axis, sideways is vertical
s = 0.6
ax.quiver(1, 0, 0, s, 0, 0, color="black", lw=1.4, arrow_length_ratio=0.2)
ax.quiver(1, 0, 0, 0, s, 0, color="black", lw=1.4, arrow_length_ratio=0.2)
ax.plot([1, 1], [0, 0], [0, qf[1]], color=PINK, lw=1.8, ls=":")
ax.text(1 + s + 0.05, 0, -0.08, "forward")
ax.text(1, s + 0.1, -0.05, "turn")
ax.text2D(0.55, 0.6, "sideways\n(forbidden)", transform=ax.transAxes, color=PINK, ha="center")

ax.set_xlim(-1.3, 2.6)
ax.set_ylim(-1.3, 1.3)
ax.set_zlim(-0.75, 1.25)
ax.set_box_aspect((3.9, 2.6, 2.0), zoom=1.35)
ax.set_axis_off()
plt.show()

*The TurtleBot's C-space drawn with the map $(x, y, \theta) \mapsto (e^x\cos\theta,\ e^x\sin\theta,\ y)$. From $q_0$, driving forward moves away from the axis and turning moves around it. Moving sideways would go straight up, which the robot cannot do. The parking path still ends at the point straight above $q_0$.*

<video
  src="https://artifacts.aegeanai.com/manim/configuration-space/TurtleBotCSpace.mp4"
  controls
  loop
  muted
  playsInline
  width="100%"
  style={{ borderRadius: 8 }}
/>

*On the left, the robot seen from above. On the right, its configuration as one point. The stack of floors opens like a book around its spine, and the last page lands on the first. A turn in place walks once around the spine. The pages at one value of $x$ form a cylinder surface, and the space is these surfaces nested around the spine. At $q_0$ the robot can drive forward or turn but not slide sideways, which points straight up. The parallel-parking path still ends straight above $q_0$.*

The vector $\dot q$ is the *configuration velocity*. All possible velocities at $q$ form the *tangent space* there. The equation $A(q)\dot q = 0$ is a *velocity constraint*. The two columns in the motion equation give the robot's *allowed directions*.

The configuration space describes where the robot can be. The velocity constraint describes which directions it can move in from each configuration.

A rigid body's configuration space also has an algebraic structure. Its elements are motions that you can compose and invert. [Lie groups and tangent spaces](/aiml-common/lectures/motion-algebras/lie-theory/index) develops this structure.

### Task space

The task space is a space in which you can express the robot's task naturally. Consider incremental sheet forming, shown below. A metal sheet is clamped in a frame. The robot pushes a tool with a spherical tip into the sheet and follows a path that gradually forms the part. The path of the tip's center determines the formed surface. Rotation of the tool about that center does not change the surface. The task space is therefore $\mathbb{R}^3$, the position $(x, y, z)$ of the tip.

If the part is formed in layers of constant depth, each layer is a contour in a plane. The task space of one layer is $\mathbb{R}^2$. If the tool must tilt so that its shank does not hit the wall of a deep part, its direction becomes part of the task. The task space is then $\mathbb{R}^3 \times S^2$.

You choose the task space when describing the task, and it does not depend on the robot. A 6-dof arm performing a task in $\mathbb{R}^3$ has three degrees of freedom left. You can use them for other purposes, such as avoiding singularities or the clamping frame.

![A robot arm pressing a tool into a clamped metal sheet.](images/sheet-forming.png)

*Incremental sheet forming.*

### Workspace

The workspace specifies the end-effector configurations that the robot can reach. The robot's structure primarily determines its workspace, independently of the task.

## References

Lynch and Park, *Modern Robotics: Mechanics, Planning, and Control* (2017)